# CrisisQ — Existing selection-policy comparison

**Attach the same whole model-containing output from `CrisisQ_nonclinical_claim_verification` under `/kaggle/input`. Then Run All.** You may instead attach the latest complete CrisisQ code/results package as a folder. The required files are `final_results.csv` and `development_acceptance.csv` in the same folder. Models and input ZIPs are not needed.

Internet OFF is sufficient; GPU OFF. Uses Python standard library and NumPy already installed in Kaggle. No package installation, training, model inference, threshold fitting or data download. This is a quick saved-table analysis, not another long inference run.

**Question:** among seeds accepted by the constrained policy, what recall/alert tradeoffs distinguish it from already-recorded fixed signed/unsigned INT8 at0.70, calibration-recall-target signed INT8, and FP32 at0.70?

All comparisons are post hoc on previously inspected legacy sets. They do not isolate the causal effect of budgets, validate data independence, or create additional algorithmic novelty. Rejected seeds have no deployed selected policy; their absence is reported rather than treated as zero errors. No policy will be chosen using these outcomes.


In [ ]:
from pathlib import Path
import json, hashlib, zipfile
INPUT_BASE=Path('/kaggle/input')
INPUT_OVERRIDE=None  # optional exact directory containing the two CSVs
OUTPUT=Path('/kaggle/working/crisisq_selection_policy_audit_v1')
OUTPUT.mkdir(parents=True,exist_ok=True)


In [ ]:
"""Post-hoc comparison of already-recorded policies; no selection or inference."""
from pathlib import Path
import csv, hashlib, json
import numpy as np

def read(path):
    with Path(path).open(newline='') as f: return list(csv.DictReader(f))
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def write(path,rows):
    if not rows: return
    with Path(path).open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
def main(source,output):
    source,output=Path(source),Path(output);output.mkdir(parents=True,exist_ok=True)
    paths=[source/'final_results.csv',source/'development_acceptance.csv']
    records,dev=map(read,paths)
    required={'arm','seed','variant','split','analysis','system','tau','recall','fpr','missed','false_alarms','n_pos','n_neg','size_mb'}
    assert records and required<=records[0].keys(), 'Unrecognized final-results schema'
    assert dev and {'arm','seed','system','accepted'}<=dev[0].keys()
    all_seeds=sorted({(r['arm'],r['seed']) for r in dev})
    accepted={(r['arm'],r['seed']) for r in dev if r['system']=='hybrid' and r['accepted'].lower()=='true'}
    # All policies and comparisons fixed in this code before computation.
    configs={'selected':('dev_selected',None,'hybrid_v2'),
             'fixed_signed_070':('fixed_0.70','qint8','hybrid_v2'),
             'fixed_unsigned_070':('fixed_0.70','q_quint8','hybrid_v2'),
             'signed_calibration_R90':('calibration_R90','qint8','hybrid'),
             'fp32_070':('fixed_0.70','fp32','hybrid_v2')}
    index={}
    for policy,(analysis,variant,system) in configs.items():
        for r in records:
            if r['analysis']!=analysis or r['system']!=system or (variant and r['variant']!=variant):continue
            k=(r['arm'],r['seed'],r['split'],policy)
            assert k not in index, f'Duplicate policy record {k}'
            for field in ['tau','recall','fpr','size_mb']:float(r[field] or 'nan')
            index[k]=r
    coverage=[];paired=[];summary=[]
    for arm in sorted({a for a,s in all_seeds}):
        seeds=[s for a,s in all_seeds if a==arm];passed=[s for s in seeds if (arm,s) in accepted]
        observed={s for a,s,split,policy in index if a==arm and policy=='selected'}
        assert observed==set(passed), f'Acceptance/selected-report mismatch: {arm}'
        coverage.append(dict(arm=arm,all_seeds=len(seeds),accepted_seeds=len(passed),rejected_seeds=len(seeds)-len(passed),rejection_handling='no fallback; no zero-loss or zero-alert imputation'))
        for split in sorted({r['split'] for r in records}):
            for baseline in configs:
                if baseline=='selected':continue
                rows=[]
                for seed in passed:
                    a=index.get((arm,seed,split,'selected'));b=index.get((arm,seed,split,baseline))
                    if not a or not b:continue
                    assert (a['n_pos'],a['n_neg'],a['history'])==(b['n_pos'],b['n_neg'],b['history'])
                    row=dict(arm=arm,seed=seed,split=split,baseline=baseline,selected_variant=a['variant'],selected_tau=float(a['tau']),baseline_variant=b['variant'],baseline_tau=float(b['tau']),history=a['history'])
                    for field in ['recall','fpr','missed','false_alarms','size_mb']:
                        av,bv=float(a[field] or 'nan'),float(b[field] or 'nan');row['selected_'+field]=av;row['baseline_'+field]=bv;row['delta_'+field]=av-bv
                    rows.append(row);paired.append(row)
                if not rows:continue
                for metric in ['recall','fpr','missed','false_alarms','size_mb']:
                    vals=np.array([r['delta_'+metric] for r in rows],float);vals=vals[np.isfinite(vals)]
                    if not len(vals):continue
                    rng=np.random.default_rng(20261115)
                    boots=vals[rng.integers(len(vals),size=(10000,len(vals)))].mean(axis=1)
                    lo,hi=np.quantile(boots,[.025,.975]) if len(vals)>1 else (np.nan,np.nan)
                    summary.append(dict(arm=arm,split=split,baseline=baseline,metric='source_negative_alert_rate' if metric=='fpr' else metric,n_seed_pairs=len(vals),mean_selected_minus_baseline=float(vals.mean()),seed_bootstrap_lo=float(lo),seed_bootstrap_hi=float(hi),positive_pairs=int((vals>1e-12).sum()),negative_pairs=int((vals< -1e-12).sum()),tied_pairs=int((abs(vals)<=1e-12).sum()),scope='post hoc; accepted seeds only; fixed messages; no message/population uncertainty'))
    write(output/'coverage.csv',coverage);write(output/'paired_policies.csv',paired);write(output/'paired_seed_summary.csv',summary)
    manifest=dict(source_sha256={p.name:sha(p) for p in paths},policies=configs,bootstrap_seed=20261115,bootstrap_samples=10000,training=False,inference=False,threshold_fitting=False,scope='post-hoc existing-policy comparison; previously inspected legacy sets; conditional on constrained acceptance',warnings=['Rejected seeds have no deployed selected policy; never imputed as zero errors.','Comparators differ in thresholds and possibly quantizer: this is a policy comparison, not an isolated budget effect.','Accepted-seed comparisons do not estimate unconditional deployment benefit.','Seed bootstrap does not measure unseen-source uncertainty.','Source-negative stress alerts are not adjudicated false positives.'])
    (output/'provenance.json').write_text(json.dumps(manifest,indent=2))
    (output/'READ_ME.txt').write_text('Existing-policy audit. Read coverage.csv before paired_seed_summary.csv.\nAll deltas are constrained-selected minus baseline. Recall/alert-rate values are fractions; multiply by100 for percentage points. Missed/alert counts are per-seed counts; size is decimal MB.\nNo policy is selected from these outcomes. No thresholds are fitted or models run. Rejection is reported as abstention from selection, not a zero-error system. Previously inspected legacy sets supply exploratory evidence only.\n')
    return summary


In [ ]:
# Discover by schema/files, never by a fixed Kaggle dataset name.
if INPUT_OVERRIDE is not None:
    SOURCE=Path(INPUT_OVERRIDE)
else:
    found=[p.parent for p in INPUT_BASE.rglob('final_results.csv')
           if (p.parent/'development_acceptance.csv').is_file()]
    unique={}
    for p in sorted(found):
        key=tuple(sha(p/name) for name in ['final_results.csv','development_acceptance.csv'])
        unique.setdefault(key,p)
    if not unique:
        raise FileNotFoundError('Attach the WHOLE completed nonclinical notebook output containing both tables. A model-only or runtime-only folder is insufficient. No retraining is requested.')
    if len(unique)!=1:
        raise RuntimeError('Different complete table pairs found. Set INPUT_OVERRIDE to the intended original run; no run is selected from its performance.')
    SOURCE=next(iter(unique.values()))
plan={'study':'existing-policy audit v1','scope':'post hoc; previously inspected legacy sets',
      'source_sha256':{name:sha(SOURCE/name) for name in ['final_results.csv','development_acceptance.csv']},
      'bootstrap_seed':20261115,'bootstrap_samples':10000,
      'code_sha256':hashlib.sha256('"""Post-hoc comparison of already-recorded policies; no selection or inference."""\nfrom pathlib import Path\nimport csv, hashlib, json\nimport numpy as np\n\ndef read(path):\n    with Path(path).open(newline=\'\') as f: return list(csv.DictReader(f))\ndef sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()\ndef write(path,rows):\n    if not rows: return\n    with Path(path).open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)\ndef main(source,output):\n    source,output=Path(source),Path(output);output.mkdir(parents=True,exist_ok=True)\n    paths=[source/\'final_results.csv\',source/\'development_acceptance.csv\']\n    records,dev=map(read,paths)\n    required={\'arm\',\'seed\',\'variant\',\'split\',\'analysis\',\'system\',\'tau\',\'recall\',\'fpr\',\'missed\',\'false_alarms\',\'n_pos\',\'n_neg\',\'size_mb\'}\n    assert records and required<=records[0].keys(), \'Unrecognized final-results schema\'\n    assert dev and {\'arm\',\'seed\',\'system\',\'accepted\'}<=dev[0].keys()\n    all_seeds=sorted({(r[\'arm\'],r[\'seed\']) for r in dev})\n    accepted={(r[\'arm\'],r[\'seed\']) for r in dev if r[\'system\']==\'hybrid\' and r[\'accepted\'].lower()==\'true\'}\n    # All policies and comparisons fixed in this code before computation.\n    configs={\'selected\':(\'dev_selected\',None,\'hybrid_v2\'),\n             \'fixed_signed_070\':(\'fixed_0.70\',\'qint8\',\'hybrid_v2\'),\n             \'fixed_unsigned_070\':(\'fixed_0.70\',\'q_quint8\',\'hybrid_v2\'),\n             \'signed_calibration_R90\':(\'calibration_R90\',\'qint8\',\'hybrid\'),\n             \'fp32_070\':(\'fixed_0.70\',\'fp32\',\'hybrid_v2\')}\n    index={}\n    for policy,(analysis,variant,system) in configs.items():\n        for r in records:\n            if r[\'analysis\']!=analysis or r[\'system\']!=system or (variant and r[\'variant\']!=variant):continue\n            k=(r[\'arm\'],r[\'seed\'],r[\'split\'],policy)\n            assert k not in index, f\'Duplicate policy record {k}\'\n            for field in [\'tau\',\'recall\',\'fpr\',\'size_mb\']:float(r[field] or \'nan\')\n            index[k]=r\n    coverage=[];paired=[];summary=[]\n    for arm in sorted({a for a,s in all_seeds}):\n        seeds=[s for a,s in all_seeds if a==arm];passed=[s for s in seeds if (arm,s) in accepted]\n        observed={s for a,s,split,policy in index if a==arm and policy==\'selected\'}\n        assert observed==set(passed), f\'Acceptance/selected-report mismatch: {arm}\'\n        coverage.append(dict(arm=arm,all_seeds=len(seeds),accepted_seeds=len(passed),rejected_seeds=len(seeds)-len(passed),rejection_handling=\'no fallback; no zero-loss or zero-alert imputation\'))\n        for split in sorted({r[\'split\'] for r in records}):\n            for baseline in configs:\n                if baseline==\'selected\':continue\n                rows=[]\n                for seed in passed:\n                    a=index.get((arm,seed,split,\'selected\'));b=index.get((arm,seed,split,baseline))\n                    if not a or not b:continue\n                    assert (a[\'n_pos\'],a[\'n_neg\'],a[\'history\'])==(b[\'n_pos\'],b[\'n_neg\'],b[\'history\'])\n                    row=dict(arm=arm,seed=seed,split=split,baseline=baseline,selected_variant=a[\'variant\'],selected_tau=float(a[\'tau\']),baseline_variant=b[\'variant\'],baseline_tau=float(b[\'tau\']),history=a[\'history\'])\n                    for field in [\'recall\',\'fpr\',\'missed\',\'false_alarms\',\'size_mb\']:\n                        av,bv=float(a[field] or \'nan\'),float(b[field] or \'nan\');row[\'selected_\'+field]=av;row[\'baseline_\'+field]=bv;row[\'delta_\'+field]=av-bv\n                    rows.append(row);paired.append(row)\n                if not rows:continue\n                for metric in [\'recall\',\'fpr\',\'missed\',\'false_alarms\',\'size_mb\']:\n                    vals=np.array([r[\'delta_\'+metric] for r in rows],float);vals=vals[np.isfinite(vals)]\n                    if not len(vals):continue\n                    rng=np.random.default_rng(20261115)\n                    boots=vals[rng.integers(len(vals),size=(10000,len(vals)))].mean(axis=1)\n                    lo,hi=np.quantile(boots,[.025,.975]) if len(vals)>1 else (np.nan,np.nan)\n                    summary.append(dict(arm=arm,split=split,baseline=baseline,metric=\'source_negative_alert_rate\' if metric==\'fpr\' else metric,n_seed_pairs=len(vals),mean_selected_minus_baseline=float(vals.mean()),seed_bootstrap_lo=float(lo),seed_bootstrap_hi=float(hi),positive_pairs=int((vals>1e-12).sum()),negative_pairs=int((vals< -1e-12).sum()),tied_pairs=int((abs(vals)<=1e-12).sum()),scope=\'post hoc; accepted seeds only; fixed messages; no message/population uncertainty\'))\n    write(output/\'coverage.csv\',coverage);write(output/\'paired_policies.csv\',paired);write(output/\'paired_seed_summary.csv\',summary)\n    manifest=dict(source_sha256={p.name:sha(p) for p in paths},policies=configs,bootstrap_seed=20261115,bootstrap_samples=10000,training=False,inference=False,threshold_fitting=False,scope=\'post-hoc existing-policy comparison; previously inspected legacy sets; conditional on constrained acceptance\',warnings=[\'Rejected seeds have no deployed selected policy; never imputed as zero errors.\',\'Comparators differ in thresholds and possibly quantizer: this is a policy comparison, not an isolated budget effect.\',\'Accepted-seed comparisons do not estimate unconditional deployment benefit.\',\'Seed bootstrap does not measure unseen-source uncertainty.\',\'Source-negative stress alerts are not adjudicated false positives.\'])\n    (output/\'provenance.json\').write_text(json.dumps(manifest,indent=2))\n    (output/\'READ_ME.txt\').write_text(\'Existing-policy audit. Read coverage.csv before paired_seed_summary.csv.\\nAll deltas are constrained-selected minus baseline. Recall/alert-rate values are fractions; multiply by100 for percentage points. Missed/alert counts are per-seed counts; size is decimal MB.\\nNo policy is selected from these outcomes. No thresholds are fitted or models run. Rejection is reported as abstention from selection, not a zero-error system. Previously inspected legacy sets supply exploratory evidence only.\\n\')\n    return summary\n'.encode()).hexdigest()}
path=OUTPUT/'analysis_plan.json'
if path.exists():
    assert json.loads(path.read_text())==plan, 'Changed source or code: preserve this run and use a new OUTPUT directory.'
else:path.write_text(json.dumps(plan,indent=2))
print('Verified source:',SOURCE)
summary=main(SOURCE,OUTPUT)
print((OUTPUT/'READ_ME.txt').read_text())
print('Coverage:',read(OUTPUT/'coverage.csv'))


In [ ]:
# Display all arms/policies. The full CSV retains every split and sign count.
for r in summary:
    if r['split'] in ['test','hn_final'] and r['metric'] in ['recall','source_negative_alert_rate']:
        print(r['arm'],r['baseline'],r['split'],r['metric'],
              'pairs=',r['n_seed_pairs'],
              'selected-baseline pp=',round(100*r['mean_selected_minus_baseline'],3),
              'seed interval=',[round(100*r[k],3) for k in ['seed_bootstrap_lo','seed_bootstrap_hi']])
REPORT=OUTPUT.with_name(OUTPUT.name+'_reports.zip')
with zipfile.ZipFile(REPORT,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTPUT.iterdir()):
        if p.is_file():z.write(p,p.name)
with zipfile.ZipFile(REPORT) as z:assert z.testzip() is None
from IPython.display import display, FileLink
print('Return this reports ZIP:',REPORT)
display(FileLink(str(REPORT)))


## Interpretation

Positive recall deltas mean more source-positive detections. Positive source-negative alert-rate deltas mean more alerts, not automatically more adjudicated errors. A policy that improves one while worsening the other does not dominate its comparator. Read the rejection coverage first.

The bootstrap resamples seed pairs only, conditioning on the same inspected messages and successful constrained selection. It is not a new-source confidence interval. Comparator thresholds/quantizers differ, so the analysis evaluates recorded policies, not a pure causal budget ablation. No new model should be chosen from this report.
